# sample_merge_append_joins

Generated from `alteryx-workflows\sample_merge_append_joins.yxmd` by compiler 2.2.0.

## How to use this notebook

1. Review and complete the source configuration.
2. Run cells from top to bottom.
3. Inspect the output previews and validate semantic parity.

> **Important:** Generated transformations are unvalidated. Confirm schemas, row counts, business rules, and destinations before promotion.


## Runtime imports

Imports and runtime helpers required by the generated PySpark transformations.


In [ ]:
from pyspark.sql import Window, functions as F


## Source ingestion

Loads and validates source datasets before transformation.


### Source ingestion 1 of 1

Produces `orders`, `customers`, `rates`.


In [ ]:
orders = spark.createDataFrame([["C001", "West", "Online", "SKU-A-100", "1200", "4"], ["C002", "East", "Store", "SKU-B-220", "840", "2"], ["C001", "West", "Online", "SKU-A-140", "640", "1"], ["C003", "North", "Partner", "SKU-C-010", "2100", "6"], ["C006", "South", "Store", "SKU-B-220", "430", "3"], ["C002", "East", "Online", "SKU-A-100", "990", "2"]], ["CustomerID", "Region", "Channel", "SKU", "Amount", "Units"])
customers = spark.createDataFrame([["C001", "Acme Retail", "Gold"], ["C002", "Northwind", "Silver"], ["C003", "Contoso", "Gold"], ["C004", "Fabrikam", "Bronze"]], ["CustomerID", "CustomerName", "Tier"])
rates = spark.createDataFrame([["0.08", "USD", "FY2026"]], ["TaxRate", "Currency", "PriceList"])


## Transformations

Applies the translated Alteryx operations in dependency order.


### Transformations 1 of 2

Produces `keep_order_fields`, `amount_and_units_1`, `amount_and_units_2`, `amount_greater_than_zero`, and 4 more.


In [ ]:
keep_order_fields = orders
amount_and_units_1 = orders.withColumn("AmountNum", F.col("Amount").cast('double'))
amount_and_units_2 = amount_and_units_1.withColumn("UnitsNum", F.col("Units").cast('double'))
amount_greater_than_zero = amount_and_units_2.filter(F.col("Amount").cast('double') > 0)
largest_amount_first = amount_greater_than_zero.orderBy(F.col("AmountNum").desc())
keep_customer_fields = customers.select(*["CustomerID", "CustomerName", "Tier"])
merge_as_new = largest_amount_first.join(keep_customer_fields, (largest_amount_first["CustomerID"] == keep_customer_fields["CustomerID"]), 'inner').drop(keep_customer_fields["CustomerID"])
append_as_new = merge_as_new.crossJoin(rates)


### Transformations 2 of 2

Produces `amount_with_tax`.


In [ ]:
amount_with_tax = append_as_new.withColumn("AmountWithTax", F.col("AmountNum") * (F.lit(1) + F.col("TaxRate").cast('double')))


## Output preview

Displays terminal DataFrames for review. Replace preview-only output with approved destination writes after validation.


### Output preview 1 of 1


In [ ]:
display(merge_as_new)
display(amount_with_tax)
